# GP - SWIM Experiments

In [1]:
import torch
import gpytorch
import numpy as np
%matplotlib inline
import matplotlib
import matplotlib.pyplot as plt

## STAGE 1: Create a TOY dataset and fit an Exact Gaussian Process 

In [2]:
torch.manual_seed(42)

# ─── 1. Create dataset ───────────────────────────────────
N_train = 100
N_test  = 300

# Input: uniform in [-3, 3]
X_train = torch.linspace(-3, 3, N_train).unsqueeze(1)  # shape (100, 1)
y_train = torch.sin(X_train.squeeze()) + 0.1 * torch.randn(N_train)

X_test  = torch.linspace(-4, 4, N_test).unsqueeze(1)   # shape (300, 1)
y_test  = torch.sin(X_test.squeeze())                   # noiseless ground truth

print(f"X_train: {X_train.shape},  y_train: {y_train.shape}")
print(f"X_test:  {X_test.shape},   y_test:  {y_test.shape}")


X_train: torch.Size([100, 1]),  y_train: torch.Size([100])
X_test:  torch.Size([300, 1]),   y_test:  torch.Size([300])


In [3]:
# ─── 2. Define GP model ──────────────────────────────────
class ExactGPModel(gpytorch.models.ExactGP):
    def __init__(self, X_train, y_train, likelihood):
        super().__init__(X_train, y_train, likelihood)
        self.mean_module  = gpytorch.means.ConstantMean()
        self.covar_module = gpytorch.kernels.ScaleKernel(
            gpytorch.kernels.RBFKernel()
        )

    def forward(self, x):
        mean  = self.mean_module(x)
        covar = self.covar_module(x)
        return gpytorch.distributions.MultivariateNormal(mean, covar) # type: ignore

In [4]:
# ─── 3. Initialize ───────────────────────────────────────
likelihood = gpytorch.likelihoods.GaussianLikelihood()
model      = ExactGPModel(X_train, y_train, likelihood)

In [5]:
# ─── 4. Train ────────────────────────────────────────────
model.train()
likelihood.train()

optimizer = torch.optim.Adam(model.parameters(), lr=0.1)
mll       = gpytorch.mlls.ExactMarginalLogLikelihood(likelihood, model)

num_iters = 100 
for i in range(num_iters):
    optimizer.zero_grad()
    loss = -mll(model(X_train), y_train) # type: ignore
    loss.backward()
    optimizer.step()

print(f"\nGP fitted successfully.")
print(f"  Length scale: {model.covar_module.base_kernel.lengthscale.item():.4f}")
print(f"  Output scale: {model.covar_module.outputscale.item():.4f}")
print(f"  Noise:        {likelihood.noise.item():.4f}")
print(f"  Mean const:   {model.mean_module.constant.item():.4f}") # type: ignore


GP fitted successfully.
  Length scale: 1.2596
  Output scale: 0.6832
  Noise:        0.0089
  Mean const:   0.1524


In [6]:
# ─── 5. Freeze GP ────────────────────────────────────────
model.eval()
likelihood.eval()
print(f"\nGP frozen. Ready for Stage 2 — pair sampling.")


GP frozen. Ready for Stage 2 — pair sampling.


## STAGE 2: GP Driven SWIM Scores

In [7]:
"""
    X_train,          # torch tensor (N, d)
    y_train,          # torch tensor (N,)
    model,            # fitted frozen GP model
    likelihood,       # fitted frozen likelihood
    M,                # number of candidate pairs
    N_pairs,          # number of pairs to select (= layer_width equivalent)
    T=3,              # number of interior points per pair
    epsilon=1e-8,     # numerical stability
    random_seed=42
"""

random_seed = 42
rng = np.random.default_rng(random_seed)
N = X_train.shape[0]

In [8]:
# ── Step 1: Sample M candidate pairs ─────────────────
# Same logic as SWIM — delta trick guarantees idx_from != idx_to
M = 100 # Update later
idx_from = rng.integers(low=0, high=N, size=M)
delta    = rng.integers(low=1, high=N-1, size=M)
idx_to   = (idx_from + delta) % N

# Select corr. values using the indices list
x_a = X_train[idx_from]   # shape (M, d)
x_b = X_train[idx_to]     # shape (M, d)
y_a = y_train[idx_from]   # shape (M,)
y_b = y_train[idx_to]   # shape (M,)

In [9]:
# ── Step 2: Create T interior points per pair ────────
# t in {1/(T+1), 2/(T+1), ..., T/(T+1)} — avoids endpoints
T = 3
t_values = torch.linspace(0, 1, T+2)[1:-1]  # shape (T,)
# x_t shape: (M, T, d)
# x_a[:, None, :] broadcasts to (M, 1, d)
x_interior = (
    x_a.unsqueeze(1) +
    t_values.view(1, T, 1) * (x_b - x_a).unsqueeze(1)
)  # (M, T, d)
# Flatten to (M*T, d) for single GP query
x_interior_flat = x_interior.reshape(M * T, -1)

In [10]:
# ── Step 3: Query frozen GP at interior points ───────
with torch.no_grad(), gpytorch.settings.fast_pred_var():
    pred         = likelihood(model(x_interior_flat))
    mu_interior  = pred.mean.reshape(M, T)      # (M, T)
    std_interior = pred.variance.sqrt().reshape(M, T)  # (M, T)

# ── Endpoint gradients: need grad through mu ──
x_a_g = x_a.detach().requires_grad_(True)  # (M, d)
x_b_g = x_b.detach().requires_grad_(True)  # (M, d)

with gpytorch.settings.fast_pred_var():
    pred_a = likelihood(model(x_a_g))
    pred_b = likelihood(model(x_b_g))
    
    mu_a  = pred_a.mean          # (M,)
    std_a = pred_a.variance.sqrt()  # (M,)
    
    mu_b  = pred_b.mean          # (M,)
    std_b = pred_b.variance.sqrt()  # (M,)

# ── Numerator: L-inf norm of gradient difference ──
grad_a = torch.autograd.grad(mu_a.sum(), x_a_g)[0]  # (M, d)
grad_b = torch.autograd.grad(mu_b.sum(), x_b_g)[0]  # (M, d)

numerator = (grad_a - grad_b).abs().max(dim=1).values  # (M,)

# ── Denominator: uncertainty at endpoints + along segment ──
epsilon = 1e-6
denominator = std_a + std_interior.sum(dim=1) + std_b + epsilon  # (M,) # dont add boundaries

# ── Scores and probabilities ──
scores      = numerator / denominator          # (M,)
probs       = scores / scores.sum()            # (M,)  sums to 1

In [11]:
scores

tensor([0.3092, 1.3311, 0.7313, 0.0910, 1.8451, 2.5379, 0.6513, 0.3774, 1.1018,
        1.0275, 2.9453, 1.6697, 0.3155, 0.1443, 0.4672, 1.3777, 0.0789, 1.4221,
        0.8127, 0.0817, 3.8840, 3.6086, 0.5392, 2.9557, 2.0383, 1.5123, 0.6715,
        2.9231, 3.5342, 3.7996, 4.5547, 0.3372, 1.1981, 3.6418, 2.3502, 3.1354,
        3.0878, 0.3220, 1.2265, 3.4800, 0.3890, 1.5525, 2.1770, 1.2584, 0.2641,
        0.9387, 0.2796, 0.2247, 1.2866, 0.5704, 2.0429, 2.4280, 0.8304, 2.2096,
        3.7722, 2.9257, 1.3369, 1.2341, 0.1498, 1.2296, 0.1069, 1.5403, 1.4667,
        2.7806, 2.4701, 1.8794, 1.4103, 1.0354, 3.4078, 0.1628, 0.8658, 0.7386,
        2.0954, 0.2177, 0.4402, 3.7205, 3.1107, 1.3345, 0.1499, 1.6617, 0.4568,
        0.1760, 1.7988, 3.2124, 2.8657, 1.3641, 1.8656, 0.0196, 2.3997, 0.0760,
        3.1133, 2.3889, 2.9255, 1.3306, 0.3590, 0.7886, 2.0231, 1.4303, 3.1236,
        0.7334], grad_fn=<DivBackward0>)

In [12]:
probs

tensor([0.0020, 0.0084, 0.0046, 0.0006, 0.0117, 0.0160, 0.0041, 0.0024, 0.0070,
        0.0065, 0.0186, 0.0106, 0.0020, 0.0009, 0.0030, 0.0087, 0.0005, 0.0090,
        0.0051, 0.0005, 0.0245, 0.0228, 0.0034, 0.0187, 0.0129, 0.0096, 0.0042,
        0.0185, 0.0223, 0.0240, 0.0288, 0.0021, 0.0076, 0.0230, 0.0148, 0.0198,
        0.0195, 0.0020, 0.0077, 0.0220, 0.0025, 0.0098, 0.0138, 0.0080, 0.0017,
        0.0059, 0.0018, 0.0014, 0.0081, 0.0036, 0.0129, 0.0153, 0.0052, 0.0140,
        0.0238, 0.0185, 0.0084, 0.0078, 0.0009, 0.0078, 0.0007, 0.0097, 0.0093,
        0.0176, 0.0156, 0.0119, 0.0089, 0.0065, 0.0215, 0.0010, 0.0055, 0.0047,
        0.0132, 0.0014, 0.0028, 0.0235, 0.0197, 0.0084, 0.0009, 0.0105, 0.0029,
        0.0011, 0.0114, 0.0203, 0.0181, 0.0086, 0.0118, 0.0001, 0.0152, 0.0005,
        0.0197, 0.0151, 0.0185, 0.0084, 0.0023, 0.0050, 0.0128, 0.0090, 0.0197,
        0.0046], grad_fn=<DivBackward0>)

In [13]:
probs.sum()

tensor(1.0000, grad_fn=<SumBackward0>)

In [31]:
# ── Step 5: Sample winning pairs ──
layer_width = 10
probs_np = probs.detach().cpu().numpy()  # multinomial needs numpy for rng.choice

selected_idx = rng.choice(
    M,                        # sample from M candidates
    size=layer_width,         # pick layer_width winners
    replace=True,             # same pair can be selected multiple times
    p=probs_np
)

# Index into your pair tensors
x_a_selected = x_a[selected_idx]  # (layer_width, d)
x_b_selected = x_b[selected_idx]  # (layer_width, d)

In [32]:
x_a_selected

tensor([[-0.2121],
        [ 1.2424],
        [-0.3333],
        [ 1.3030],
        [-0.3939],
        [ 1.2424],
        [ 1.7273],
        [-0.2727],
        [ 0.2727],
        [ 2.5758]])

In [33]:
x_b_selected

tensor([[-1.6667],
        [ 2.9394],
        [-2.5758],
        [-1.7273],
        [-2.3333],
        [ 2.9394],
        [ 0.7576],
        [-2.9394],
        [ 1.0606],
        [-0.6364]])

In [34]:
# ── Step 7: Sample GP posterior functions over selected segments ──

# Create dense interior points for each selected pair (for smooth function)
T_sample = 200  # more points for a smooth curve equivalent to 50
t_dense  = torch.linspace(0, 1, T_sample)  # (T_sample,)

# Interior points for selected pairs only
x_segments = (
    x_a_selected.unsqueeze(1) +
    t_dense.view(1, T_sample, 1) * (x_b_selected - x_a_selected).unsqueeze(1)
)  # (layer_width, T_sample, d)

# Flatten for GP query
x_segments_flat = x_segments.reshape(layer_width * T_sample, -1)  # (layer_width*T_sample, d)

# Get posterior distribution over these points
with gpytorch.settings.fast_pred_var():
    pred_segments = likelihood(model(x_segments_flat))

# Reshape mean and covariance for sampling
# We need to sample per segment separately
sampled_functions = []

for i in range(layer_width):
    # Points for this segment
    x_seg_i = x_segments[i]  # (T_sample, d)
    
    with gpytorch.settings.fast_pred_var():
        pred_i = likelihood(model(x_seg_i))
    
    # Sample one function from the posterior
    f_sample = pred_i.mean  # (T_sample,)
    sampled_functions.append(f_sample)

sampled_functions = torch.stack(sampled_functions)  # (layer_width, T_sample)

In [35]:
x_segments.shape, x_segments[0]

(torch.Size([10, 200, 1]),
 tensor([[-0.2121],
         [-0.2194],
         [-0.2267],
         [-0.2340],
         [-0.2414],
         [-0.2487],
         [-0.2560],
         [-0.2633],
         [-0.2706],
         [-0.2779],
         [-0.2852],
         [-0.2925],
         [-0.2998],
         [-0.3071],
         [-0.3145],
         [-0.3218],
         [-0.3291],
         [-0.3364],
         [-0.3437],
         [-0.3510],
         [-0.3583],
         [-0.3656],
         [-0.3729],
         [-0.3802],
         [-0.3875],
         [-0.3949],
         [-0.4022],
         [-0.4095],
         [-0.4168],
         [-0.4241],
         [-0.4314],
         [-0.4387],
         [-0.4460],
         [-0.4533],
         [-0.4606],
         [-0.4679],
         [-0.4753],
         [-0.4826],
         [-0.4899],
         [-0.4972],
         [-0.5045],
         [-0.5118],
         [-0.5191],
         [-0.5264],
         [-0.5337],
         [-0.5410],
         [-0.5483],
         [-0.5557],
         [-0.

In [36]:
sampled_functions[0]

tensor([-0.2587, -0.2660, -0.2734, -0.2807, -0.2879, -0.2952, -0.3024, -0.3096,
        -0.3167, -0.3238, -0.3309, -0.3379, -0.3449, -0.3518, -0.3588, -0.3657,
        -0.3725, -0.3793, -0.3861, -0.3928, -0.3995, -0.4062, -0.4128, -0.4194,
        -0.4259, -0.4324, -0.4389, -0.4453, -0.4517, -0.4580, -0.4643, -0.4706,
        -0.4768, -0.4830, -0.4891, -0.4952, -0.5012, -0.5072, -0.5132, -0.5191,
        -0.5250, -0.5308, -0.5366, -0.5423, -0.5480, -0.5536, -0.5592, -0.5648,
        -0.5703, -0.5758, -0.5812, -0.5866, -0.5920, -0.5972, -0.6025, -0.6077,
        -0.6128, -0.6179, -0.6230, -0.6280, -0.6330, -0.6379, -0.6428, -0.6477,
        -0.6525, -0.6572, -0.6619, -0.6666, -0.6712, -0.6757, -0.6803, -0.6847,
        -0.6892, -0.6936, -0.6979, -0.7022, -0.7065, -0.7107, -0.7149, -0.7190,
        -0.7231, -0.7271, -0.7311, -0.7351, -0.7390, -0.7428, -0.7467, -0.7505,
        -0.7542, -0.7579, -0.7615, -0.7651, -0.7687, -0.7722, -0.7757, -0.7792,
        -0.7826, -0.7859, -0.7893, -0.79

In [37]:
sampled_functions

tensor([[-0.2587, -0.2660, -0.2734,  ..., -0.9454, -0.9454, -0.9453],
        [ 0.9781,  0.9801,  0.9821,  ...,  0.2552,  0.2480,  0.2409],
        [-0.3765, -0.3869, -0.3973,  ..., -0.5832, -0.5730, -0.5627],
        ...,
        [-0.3187, -0.3317, -0.3446,  ..., -0.1903, -0.1733, -0.1562],
        [ 0.2677,  0.2721,  0.2764,  ...,  0.9128,  0.9146,  0.9163],
        [ 0.5342,  0.5463,  0.5584,  ..., -0.6005, -0.6120, -0.6232]],
       grad_fn=<StackBackward0>)

In [38]:
sampled_functions.shape

torch.Size([10, 200])

In [39]:
# ── Step 8: Interpolate edge functions at X_train and X_test ──
H_train = torch.zeros(N_train, layer_width)
H_test  = torch.zeros(N_test,  layer_width)

for i in range(layer_width):
    seg_x = x_segments[i, :, 0].detach().numpy()   # (T_sample,) — x positions of edge i
    seg_f = sampled_functions[i].detach().numpy()   # (T_sample,) — φi values at those x positions

    x_train_np = X_train[:, 0].detach().numpy()    # (N_train,)
    x_test_np  = X_test[:, 0].detach().numpy()     # (N_test,)

    # For each training point: interpolate φi(x) from the lookup table
    H_train[:, i] = torch.tensor(np.interp(x_train_np, seg_x, seg_f))
    H_test[:, i]  = torch.tensor(np.interp(x_test_np,  seg_x, seg_f))

In [40]:
layer_width

10

In [41]:
x_train_np

array([-3.        , -2.939394  , -2.878788  , -2.8181818 , -2.7575758 ,
       -2.6969697 , -2.6363635 , -2.5757575 , -2.5151515 , -2.4545455 ,
       -2.3939395 , -2.3333333 , -2.2727273 , -2.2121212 , -2.151515  ,
       -2.090909  , -2.030303  , -1.969697  , -1.9090909 , -1.8484848 ,
       -1.7878788 , -1.7272727 , -1.6666666 , -1.6060605 , -1.5454545 ,
       -1.4848485 , -1.4242424 , -1.3636363 , -1.3030303 , -1.2424242 ,
       -1.1818181 , -1.121212  , -1.060606  , -0.99999994, -0.9393939 ,
       -0.8787878 , -0.81818175, -0.7575757 , -0.6969696 , -0.63636357,
       -0.5757575 , -0.51515144, -0.45454538, -0.39393932, -0.33333325,
       -0.2727272 , -0.21212113, -0.15151507, -0.090909  , -0.03030294,
        0.03030294,  0.090909  ,  0.15151507,  0.21212113,  0.2727272 ,
        0.33333325,  0.39393932,  0.45454538,  0.51515144,  0.5757575 ,
        0.63636357,  0.6969696 ,  0.7575757 ,  0.81818175,  0.8787878 ,
        0.9393939 ,  0.99999994,  1.060606  ,  1.121212  ,  1.18

In [42]:
H_train

tensor([[-0.2587,  0.9781, -0.3765,  0.9907, -0.4316,  0.9781,  0.9757, -0.3187,
          0.2677,  0.5342],
        [-0.2587,  0.9781, -0.3765,  0.9907, -0.4316,  0.9781,  0.9757, -0.3187,
          0.2677,  0.5342],
        [-0.2587,  0.9781, -0.3765,  0.9907, -0.4316,  0.9781,  0.9757, -0.1562,
          0.2677,  0.5342],
        [-0.2587,  0.9781, -0.3765,  0.9907, -0.4316,  0.9781,  0.9757, -0.1562,
          0.2677,  0.5342],
        [-0.2587,  0.9781, -0.3765,  0.9907, -0.4316,  0.9781,  0.9757, -0.1562,
          0.2677,  0.5342],
        [-0.2587,  0.9781, -0.3765,  0.9907, -0.4316,  0.9781,  0.9757, -0.1562,
          0.2677,  0.5342],
        [-0.2587,  0.9781, -0.3765,  0.9907, -0.4316,  0.9781,  0.9757, -0.1562,
          0.2677,  0.5342],
        [-0.2587,  0.9781, -0.3765,  0.9907, -0.4316,  0.9781,  0.9757, -0.1562,
          0.2677,  0.5342],
        [-0.2587,  0.9781, -0.5627,  0.9907, -0.4316,  0.9781,  0.9757, -0.1562,
          0.2677,  0.5342],
        [-0.2587,  

In [43]:
H_train.shape

torch.Size([100, 10])

In [44]:
# ── Step 9: OLS — solve for output layer ──
H_train_b = torch.cat([H_train, torch.ones(N_train, 1)], dim=1)  # (N_train, layer_width+1)
H_test_b  = torch.cat([H_test,  torch.ones(N_test,  1)], dim=1)  # (N_test,  layer_width+1)

result = torch.linalg.lstsq(H_train_b, y_train.unsqueeze(1))
W_out  = result.solution  # (layer_width+1, 1)

# ── Step 10: Predict and evaluate ──
y_pred = H_test_b @ W_out
mse    = ((y_pred.squeeze() - y_test) ** 2).mean()
print(f"\nTest MSE: {mse.item():.6f}")


Test MSE: 0.087686


In [45]:
# Baseline 1: GP posterior mean directly
with torch.no_grad():
    gp_pred = likelihood(model(X_test)).mean
    gp_mse  = ((gp_pred - y_test) ** 2).mean()
    print(f"GP baseline MSE:   {gp_mse.item():.6f}")

# Baseline 2: predicting mean of y_train
# double check
mean_pred = y_train.mean().expand(N_test)
mean_mse  = ((mean_pred - y_test) ** 2).mean()
print(f"Mean baseline MSE: {mean_mse.item():.6f}")

GP baseline MSE:   0.012779
Mean baseline MSE: 0.438664


In [46]:
# Relative L2 error = ||y_pred - y_test||_2 / ||y_test||_2
rel_l2 = torch.norm(y_pred.squeeze() - y_test) / torch.norm(y_test)
print(f"Relative L2 error: {rel_l2.item():.6f}")

# For all baselines too
gp_rel_l2   = torch.norm(gp_pred - y_test) / torch.norm(y_test)
mean_rel_l2 = torch.norm(mean_pred - y_test) / torch.norm(y_test)

print(f"GP baseline relative L2:   {gp_rel_l2.item():.6f}")
print(f"Mean baseline relative L2: {mean_rel_l2.item():.6f}")

Relative L2 error: 0.447112
GP baseline relative L2:   0.170689
Mean baseline relative L2: 1.000041


In [47]:
# take different function types
# describe and write in your thesis